In [1]:
import numpy as np


In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import jax
import functools
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import rapids_singlecell as rsc
import flax.linen as nn
import optax
import cellflow
from cellflow.model import CellFlow
import cellflow.preprocessing as cfpp
from cellflow.utils import match_linear
from cellflow.plotting import plot_condition_embedding
from cellflow.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca, reconstruct_pca
from cellflow.metrics import compute_r_squared, compute_e_distance

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [51]:
def compute_frechet_var(x: dict[int, np.ndarray]):
    pairwise_e_dists = []
    denominator = 2 * len(x)**2
    for s1 in x.values():
        for s2 in x.values():
            pairwise_e_dists.append(compute_e_distance(s1, s2)**2)
    return np.sum(np.array(pairwise_e_dists))/denominator

def compute_metrics(adata_ref: ad.AnnData, adata_pred_all_samples: ad.AnnData, donor_deg_dict: dict, adata_ood_true: ad.AnnData, adata_ctrl: ad.AnnData, n_neighbors: int=1, cell_type_col: str = "cell_type_new", min_cells_for_dist_metrics: int = 50) -> dict:
    
    compute_wknn(ref_adata=adata_ref, query_adata=adata_pred_all_samples, n_neighbors=n_neighbors, ref_rep_key="X_pca", query_rep_key="X_pca_for_ct_transfer")
    transfer_labels(query_adata=adata_pred_all_samples, ref_adata=adata_ref, label_key=cell_type_col)
    
    dict_to_log = {}
    
    ood_e_distances = []
    decoded_ood_r_squareds = []
    mean_decoded_r_sq_per_cell_types = []
    mean_e_distance_per_cell_types = []
    mean_deg_r_sq_per_cell_types = []

    for i in range(5):
        print(i)
        adata_pred = adata_pred_all_samples[adata_pred_all_samples.obs["sample"]==i]
        # standard metrics
        print(len(adata_pred), len(adata_ood_true))
        ood_e_distance = compute_e_distance(adata_ood_true.obsm["X_pca"], adata_pred.obsm["X_pca"])
        decoded_ood_r_squared = compute_r_squared(adata_ood_true.X.toarray(), adata_pred.layers["X_recon"])

        r_sq = {}
        e_distance = {}
        deg_r_sq = {}
        
        for ct_cyto in donor_deg_dict.keys(): 
            cell_type = ct_cyto.split("_")[1]
            adata_true_ct = adata_ood_true[(adata_ood_true.obs[f"{cell_type_col}"]==cell_type)]
            adata_pred_ct = adata_pred[adata_pred.obs[f"{cell_type_col}_transfer"]==cell_type]
            if ((adata_true_ct.n_obs == 0) or adata_pred_ct.n_obs==0):
                continue
            if adata_pred_ct.n_obs == 0:
                continue
            dist_true_decoded = adata_true_ct.X.toarray()
            dist_pred_decoded = adata_pred_ct.X
            dist_true = adata_true_ct.obsm["X_pca"]
            dist_pred = adata_pred_ct.obsm["X_pca"]
            r_sq[f"decoded_r_squared_{cell_type}"] = compute_r_squared(dist_true_decoded, dist_pred_decoded)
            e_distance[f"e_distance_{cell_type}"] = compute_e_distance(dist_true, dist_pred)
            
            deg_mask = [True if el in donor_deg_dict[ct_cyto] else False for el in adata_ood_true.var_names]
            deg_true_decoded = adata_true_ct[:,deg_mask].X.toarray()
            deg_pred_decoded = adata_pred_ct[:,deg_mask].X
            deg_r_sq[f"deg_decoded_r_squared_{cell_type}"] = compute_r_squared(deg_true_decoded, deg_pred_decoded)

        ood_e_distances.append(ood_e_distance)
        decoded_ood_r_squareds.append(decoded_ood_r_squared)
        mean_decoded_r_sq_per_cell_types.append(np.mean(list(r_sq.values())))
        mean_e_distance_per_cell_types.append(np.mean(list(e_distance.values())))
        mean_deg_r_sq_per_cell_types.append(np.mean(list(deg_r_sq.values())))

    # metrics to return
    dict_to_log["ood_e_distance"] = np.mean(ood_e_distances)
    dict_to_log["decoded_ood_r_squared"] = np.mean(decoded_ood_r_squareds)

    dict_to_log["mean_decoded_r_sq_per_cell_type"] = np.mean(mean_decoded_r_sq_per_cell_types)
    dict_to_log["mean_e_distance_per_cell_type"] = np.mean(mean_e_distance_per_cell_types)
    dict_to_log["mean_deg_r_sq_per_cell_type"] = np.mean(mean_deg_r_sq_per_cell_types)

    distr_dict = {}
    distr_dict_per_cell_type = {}
    gex_dict = {}
    gex_dict_per_cell_type = {}
    deg_gex_per_cell_type = {}
    for sample_idx in range(5):
        adata_pred = adata_pred_all_samples[adata_pred_all_samples.obs["sample"]==sample_idx]
        distr_dict[sample_idx] = adata_pred.obsm["X_pca"]
        gex_dict[sample_idx] = np.mean(adata_pred.X, axis=0)
        for ct in adata_pred.obs[f"{cell_type_col}_transfer"].unique():
            adata_pred_ct = adata_pred[adata_pred.obs[f"{cell_type_col}_transfer"]==ct]
            if ct not in distr_dict_per_cell_type:
                distr_dict_per_cell_type[ct] = {}
            distr_dict_per_cell_type[ct][sample_idx] = adata_pred_ct.obsm["X_pca"]
            if ct not in gex_dict_per_cell_type:
                gex_dict_per_cell_type[ct] = {}
            gex_dict_per_cell_type[ct][sample_idx] = np.mean(adata_pred_ct.X, axis=0)
            if ct not in deg_gex_per_cell_type:
                deg_gex_per_cell_type[ct] = {}
            deg_mask = [True if el in donor_deg_dict[ct_cyto] else False for el in adata_ood_true.var_names]
            deg_gex_per_cell_type[ct][sample_idx] = np.mean(adata_pred_ct[:,deg_mask].X, axis=0)
            

    dict_to_log["frechet_variance"] = compute_frechet_var(distr_dict)
    dict_to_log["var_mean_gex"] = np.var(np.stack(list(gex_dict.values())), axis=0).mean()
    f_var_per_cell_types = []

    for ct, distr_ct in distr_dict_per_cell_type.items():
        f_var_ct = compute_frechet_var(distr_ct)
        f_var_per_cell_types.append(f_var_ct)
        dict_to_log[f"frechet_variance_{ct}"] = f_var_ct

    dict_to_log["mean_frechet_variance_per_cell_type"] = np.mean(f_var_per_cell_types)
    
    var_mean_gex_per_cell_types = []
    var_mean_deg_gex_per_cell_types = []
    for ct, gex_ct in gex_dict_per_cell_type.items():
        var_mean_gex_ct = np.var(np.stack(list(gex_ct.values())), axis=0).mean()
        var_mean_gex_per_cell_types.append(var_mean_gex_ct)
        dict_to_log[f"var_mean_gex_{ct}"] = var_mean_gex_ct
        deg_gex_ct = deg_gex_per_cell_type[ct]
        var_mean_deg_gex_ct = np.var(np.stack(list(deg_gex_ct.values())), axis=0).mean()
        var_mean_deg_gex_per_cell_types.append(var_mean_deg_gex_ct)
        dict_to_log[f"var_mean_deg_gex_{ct}"] = var_mean_deg_gex_ct

    dict_to_log["mean_var_mean_gex_per_cell_type"] = np.mean(var_mean_gex_per_cell_types)
    dict_to_log["mean_var_mean_deg_gex_per_cell_type"] = np.mean(var_mean_deg_gex_per_cell_types)

    
    return dict_to_log, distr_dict, gex_dict # remove last three



In [3]:
cytokine_held_out = "TWEAK"
idx_given_cytokine = "7"
control_key = "is_control"

adata_base = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_cytokine/adata_base_{cytokine_held_out}.h5ad")
adata_rest = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_cytokine/adata_rest_{cytokine_held_out}.h5ad")
cellflow.preprocessing.centered_pca(adata_base, n_comps=100, method="rapids", keep_centered_data=False)
cellflow.preprocessing.project_pca(query_adata=adata_rest, ref_adata=adata_base)


In [4]:
idx_given_cytokine = "7"

In [5]:
donors_to_impute = adata_rest.uns["split_info"][idx_given_cytokine]["donors_to_impute"]
donors_to_train_data = adata_rest.uns["split_info"][idx_given_cytokine]["donors_to_train_data"]
adata_to_append = adata_rest[adata_rest.obs["donor"].isin(donors_to_train_data)]
adata_train = ad.concat((adata_base, adata_to_append))
adata_train.uns = adata_base.uns.copy()
adata_ood_perturbed = adata_rest[adata_rest.obs["donor"].isin(donors_to_impute)]

adata_ctrl = adata_train[adata_train.obs[control_key].to_numpy()]

adata_ctrl_subsetted = []
for donor in adata_ctrl.obs["donor"].unique():
    adata_tmp = adata_ctrl[adata_ctrl.obs["donor"]==donor]
    if adata_tmp.n_obs > 10000:
        sc.pp.subsample(adata_tmp, n_obs=10000)
    adata_ctrl_subsetted.append(adata_tmp)
adata_ctrl = ad.concat(adata_ctrl_subsetted)


adata_train.uns = adata_train.uns.copy()
adata_ctrl.uns = adata_train.uns.copy()
adata_ood_perturbed.uns = adata_train.uns.copy()


cf = CellFlow(adata_train, solver="otfm")

perturbation_covariates = {"cytokines": ["cytokine"]}
split_covariates = ["donor"]

cf.prepare_data(
    sample_rep="X_pca",
    control_key=control_key,
    perturbation_covariates=perturbation_covariates,
    perturbation_covariate_reps={"cytokines": "esm2_embeddings"},
    sample_covariates=("donor",),
    sample_covariate_reps={"donor": "donor_one_hot"},
    split_covariates=split_covariates,
)



[########################################] | 100% Completed | 103.07 ms
[########################################] | 100% Completed | 2.61 sms
[########################################] | 100% Completed | 208.95 ms


In [6]:
match_fn = functools.partial(
    match_linear,
    epsilon=10.0, #config_dict["model"]["epsilon"],
    scale_cost="mean",
    tau_a=1.0, #config_dict["model"]["tau_a"],
    tau_b=1.0, #config_dict["model"]["tau_b"]
)
#optimizer = optax.MultiSteps(optax.adam(config_dict["model"]["learning_rate"]), config_dict["model"]["multi_steps"])
#flow = {config_dict["model"]["flow_type"]: config_dict["model"]["flow_noise"]}
optimizer = optax.MultiSteps(optax.adam(1e-4), 20)
flow = {"constant_noise": 0.1}


#layers_before_pool = config_dict["model"]["layers_before_pool"]
#layers_after_pool = config_dict["model"]["layers_after_pool"]

layers_before_pool = {
    "cytokines": {"layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.5}, # cytokine_treatment replaced by cytokines 
    "donor": {"layer_type": "mlp", "dims": [256, 256], "dropout_rate": 0.0},
}

layers_after_pool = {
    "layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.0,
}

In [7]:
cf.prepare_model(
    condition_mode="stochastic",
    regularization=0.01,
    pooling="attention_token",
    pooling_kwargs={},
    layers_before_pool=layers_before_pool,
    layers_after_pool=layers_after_pool,
    condition_embedding_dim=256,
    cond_output_dropout=0.9,
    condition_encoder_kwargs={},
    pool_sample_covariates=True,
    time_freqs=1024,
    time_encoder_dims=[1024, 1024, 1024],
    time_encoder_dropout=0.0,
    hidden_dims=[2048, 2048, 2048],
    hidden_dropout=0.0,
    conditioning="concatenation",
    decoder_dims=[4096, 4096, 4096],
    vf_act_fn=nn.silu,
    vf_kwargs=None,
    probability_path={"constant_noise": 0.5},
    match_fn=match_fn,
    optimizer=optax.MultiSteps(optax.adam(5e-5), 20),
    solver_kwargs={},
    layer_norm_before_concatenation=False,
    linear_projection_before_concatenation=False,
)

    


In [14]:
adata_test = adata_ood_perturbed
adatas_train_subsampled = []
for cond in adata_train.obs["condition"].unique(): # as we have many conditions, this might take a few minutes
    adatas_train_subsampled.append(sc.pp.subsample(adata_train[adata_train.obs["condition"]==cond], n_obs=1000, copy=True))

adata_train_for_validation = ad.concat(adatas_train_subsampled)

adatas_test_subsampled = []
for cond in adata_test.obs["condition"].unique():
    adatas_test_subsampled.append(sc.pp.subsample(adata_test[adata_test.obs["condition"]==cond], n_obs=2000, copy=True))

for cond in adata_ctrl.obs["condition"].unique():
    adatas_test_subsampled.append(sc.pp.subsample(adata_ctrl[adata_ctrl.obs["condition"]==cond], n_obs=2000, copy=True))
adata_test_for_validation = ad.concat(adatas_test_subsampled)

In [12]:
adata_ctrl.obs["condition"].unique()

['Donor10_PBS', 'Donor11_PBS', 'Donor12_PBS', 'Donor1_PBS', 'Donor2_PBS', ..., 'Donor5_PBS', 'Donor6_PBS', 'Donor7_PBS', 'Donor8_PBS', 'Donor9_PBS']
Length: 12
Categories (12, object): ['Donor1_PBS', 'Donor2_PBS', 'Donor3_PBS', 'Donor4_PBS', ..., 'Donor9_PBS', 'Donor10_PBS', 'Donor11_PBS', 'Donor12_PBS']

In [13]:
cond

'Donor10_PBS'

In [15]:
adata_test_for_validation.uns = adata_test.uns.copy()

cf.prepare_validation_data(
    adata_test_for_validation,
    name="test",
    n_conditions_on_log_iteration=None,
    n_conditions_on_train_end=None,
)

[########################################] | 100% Completed | 101.82 ms
[########################################] | 100% Completed | 101.52 ms
[########################################] | 100% Completed | 100.86 ms


In [20]:
from cellflow.data._dataloader import OOCTrainSampler, PredictionSampler, TrainSampler, ValidationSampler
validation_loaders = {k: ValidationSampler(v) for k, v in cf.validation_data.items() if k != "predict_kwargs"}

In [35]:
validation_loaders.keys()

dict_keys(['train', 'test'])

In [25]:
batch = validation_loaders["test"].sample(mode="on_log_iteration")

In [28]:
len(batch["condition"])

9

In [131]:
import scipy 
from typing import Any
from cellflow.training import ComputationCallback
from cellflow._types import ArrayLike
from cellflow.solvers import _genot, _otfm
import jax.tree_util as jtu
import wandb
import numpy as np

def wandb_scatter(x, y, *, x_name: str, y_name: str, title: str, step: int | None = None):
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)

    table = wandb.Table(data=[[float(xi), float(yi)] for xi, yi in zip(x, y)],
                        columns=[x_name, y_name])

    plot = wandb.plot.scatter(table, x_name, y_name, title=title)
    payload = {title: plot}
    if step is None:
        wandb.log(payload)
    else:
        wandb.log(payload, step=step)
        
def compute_frechet_var(x: dict[int, np.ndarray]):
    pairwise_e_dists = []
    denominator = 2 * len(x)**2
    for s1 in x.values():
        for s2 in x.values():
            pairwise_e_dists.append(compute_e_distance(s1, s2)**2)
    return np.sum(np.array(pairwise_e_dists))/denominator
    
class VarCallback(ComputationCallback):
    def __init__(self, val_data: dict[str, ValidationSampler], ref_adata: ad.AnnData, n_draws: int = 10, predict_kwargs: dict[str, Any] = {}):
        self.n_draws = n_draws
        self.val_data = val_data
        self.pcs = ref_adata.varm["PCs"]
        self.means = ref_adata.varm["X_mean"]
        self.reconstruct_data = lambda x: x @ np.transpose(self.pcs) + np.transpose(self.means)
        self.predict_kwargs = predict_kwargs

    def on_train_begin(self, *args, **kwargs):
        pass

    def on_train_end(self, *args, **kwargs):
        return self.on_log_iteration(valid_source_data, valid_true_data, valid_pred_data, solver)

    def on_log_iteration(
        self, 
        valid_source_data: dict[str, dict[str, ArrayLike]],
        valid_true_data: dict[str, dict[str, ArrayLike]],
        valid_pred_data: dict[str, dict[str, ArrayLike]],
        solver: _otfm.OTFlowMatching | _genot.GENOT,
    ) -> dict[str, float]:
        valid_source_data: dict[str, dict[str, ArrayLike]] = {}
        valid_pred_data: dict[str, dict[str, ArrayLike]] = {}
        valid_true_data: dict[str, dict[str, ArrayLike]] = {}

        
        metrics = {}
        for val_key, vdl in self.val_data.items():
            mean_e_dists = []
            mean_r_sqs = []
            frechet_var = []
            var_gex = []
            batch = vdl.sample(mode="on_log_iteration")
            for cond in batch["source"].keys():
                e_distances = []
                r_sqs = []
                src = batch["source"][cond]
                condition = batch["condition"][cond]
                true_tgt = batch["target"][cond]
                valid_pred_data[cond] = {}
                for i in range(self.n_draws):
                    valid_pred_data[cond][i] = solver.predict(src, condition=condition,rng=jax.random.PRNGKey(i), **self.predict_kwargs)
                frechet_var.append(compute_frechet_var(valid_pred_data[cond]))
                for i in range(self.n_draws):
                    e_distances.append(compute_e_distance(true_tgt, valid_pred_data[cond][i]))
                valid_true_data_decoded = self.reconstruct_data(true_tgt)
                valid_pred_data_decoded = jtu.tree_map(self.reconstruct_data, valid_pred_data[cond])
                var_gex.append(np.var(np.stack(list(valid_pred_data_decoded.values())), axis=0).mean())
                for i in range(self.n_draws):
                    r_sqs.append(compute_r_squared(valid_true_data_decoded, valid_pred_data_decoded[i]))
                
                mean_r_sqs.append(np.mean(r_sqs))
                mean_e_dists.append(np.mean(e_distances))
                print(mean_r_sqs, r_sqs)

            metrics[f"{val_key}_e_dist"] = np.mean(mean_e_dists)
            metrics[f"{val_key}_r_sq"] = np.mean(mean_r_sqs)
            metrics[f"{val_key}_pearson_calibration"] = np.corrcoef(np.array(mean_e_dists), np.array(frechet_var))[0,1]
            metrics[f"{val_key}_spearman_calibration"] = scipy.stats.spearmanr(np.array(mean_e_dists), np.array(frechet_var)).statistic
            metrics[f"{val_key}_pearson_decoded_calibration"] = np.corrcoef(np.array(mean_r_sqs), -np.array(var_gex))[0,1]
            metrics[f"{val_key}_spearman_decoded_calibration"] = scipy.stats.spearmanr(np.array(mean_r_sqs), -np.array(var_gex)).statistic

            wandb_scatter(
                x=frechet_var,
                y=mean_e_dists,
                x_name="frechet_var",
                y_name="mean_e_dist",
                title=f"{val_key}/calibration_e_dist_vs_frechet_var",
            )
            
            wandb_scatter(
                x=var_gex,
                y=mean_r_sqs,
                x_name="var_gex",
                y_name="mean_r_sq",
                title=f"{val_key}/decoded_calibration_r_sq_vs_var_gex",
            )

            
        return metrics

In [132]:
class DummySolver:
    def __init__(self):
        pass

    def predict(*args, **kwargs):
        return np.random.rand(20,100)

In [133]:
#cb = VarCallback({"test": ValidationSampler(cf.validation_data["test"])}, ref_adata = adata_base, n_draws=10)

In [149]:
adata_ood = ad.concat((adata_ctrl, adata_ood_perturbed))
adata_ood.uns = adata_train.uns.copy()

cb = VarCallback({"test": ValidationSampler(cf.validation_data["test"])}, ref_adata = adata_base, n_draws=10)
wandb_callback = cellflow.training.WandbLogger(project="pbmc_with_uncertainty", out_dir="/home/icb/dominik.klein/tmp", config={})


In [151]:
callbacks = [cb, wandb_callback]

cf.train(
        num_iterations=100_000,
        batch_size=1024,
        callbacks=callbacks,
        valid_freq=25_000,
    )

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: mucdk (modality_translation) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


 25%|██▍       | 24996/100000 [08:19<20:47, 60.12it/s] 

[0.9884964736295834] [0.9883533275840586, 0.9883549724415056, 0.9893755525168324, 0.9882072080114116, 0.9883576693637146, 0.9882184614887876, 0.9884528402983436, 0.9884408618020222, 0.9886806859618674, 0.9885231568272894]
[0.9884964736295834, 0.9830747058600077] [0.9826221569842432, 0.9828293945606901, 0.9835838229862737, 0.9825420562383234, 0.9829789549130031, 0.983353353974559, 0.9834236027872869, 0.9831550286309724, 0.9830688110450032, 0.9831898764797213]
[0.9884964736295834, 0.9830747058600077, 0.9847527156173568] [0.9846860254453389, 0.9847121822214089, 0.9851126908799379, 0.9844657801891535, 0.9847122686204557, 0.9845015920412055, 0.9848745246090929, 0.9848387758859182, 0.9848914988806708, 0.9847318174003855]
[0.9884964736295834, 0.9830747058600077, 0.9847527156173568, 0.9908179940866642] [0.99029839485233, 0.9906420679540255, 0.9913801399772565, 0.9900612854365181, 0.9909026056736118, 0.9908737595839195, 0.9908640207301352, 0.9908657208516165, 0.9912854487321314, 0.9910064970750

 50%|█████     | 50001/100000 [1:04:18<13:28, 61.84it/s, loss=7.38]    

[0.9886204105144536] [0.988943043758926, 0.9886559264183001, 0.9887627296798525, 0.9888325811079847, 0.9874918691609891, 0.9886437622231191, 0.9887662210653213, 0.9887726744318922, 0.9885647954569022, 0.9887705018412488]
[0.9886204105144536, 0.975808575856723] [0.9761209747998394, 0.9763230457170136, 0.9762775556737288, 0.9760002876013848, 0.9725595679062464, 0.9760955670704224, 0.9759727978617442, 0.9755094670431393, 0.9767877710111051, 0.9764387238826049]
[0.9886204105144536, 0.975808575856723, 0.9815795063247169] [0.9818872729253875, 0.981885827400007, 0.9813020308137658, 0.9821216878966441, 0.9821071028302575, 0.9812385696556439, 0.9813266932530148, 0.9815212206908656, 0.9812277748586762, 0.9811768829229072]
[0.9886204105144536, 0.975808575856723, 0.9815795063247169, 0.9889279204004978] [0.9890537491229102, 0.9889284499682373, 0.9889343381603499, 0.9892073331449596, 0.989570818639003, 0.9884752748284898, 0.9886779677611545, 0.9890622040778002, 0.9886981839910621, 0.9886708843110104

...

[0.9788703079717029] [0.9777138737179691, 0.9793173530732838, 0.9784261683237807, 0.978924359083478, 0.9793234540729042, 0.9789967314543087, 0.9787828018795313, 0.9791462681402427, 0.9787011262031682, 0.9793709437683621]
[0.9788703079717029, 0.9806261108228617] [0.9793667534647055, 0.9813957952605237, 0.9799534395442826, 0.9805437568498164, 0.9810158834401833, 0.980923887367453, 0.9806806317249236, 0.9809527643402088, 0.9803600307378941, 0.9810681654986246]
[0.9788703079717029, 0.9806261108228617, 0.9871528357782129] [0.9872961088820721, 0.9868316297881018, 0.9873950375603884, 0.9870377579809979, 0.9871601073453841, 0.9871282372344798, 0.9872083696240301, 0.9871385743212555, 0.9870686938152179, 0.987263841230202]
[0.9788703079717029, 0.9806261108228617, 0.9871528357782129, 0.981796261522349] [0.9815068451040233, 0.9819498846291783, 0.9817179639288203, 0.9816233789518548, 0.9817125521378615, 0.9819912313292785, 0.9817963091949077, 0.9819304114726985, 0.9817897114783053, 0.98194432699656

 75%|███████▌  | 75004/100000 [2:48:42<875:40:36, 126.12s/it, loss=6.27]

[0.9788703079717029, 0.9806261108228617, 0.9871528357782129, 0.981796261522349, 0.9824946912727868, 0.9741044181801769, 0.9891524036584768, 0.9879817892967646, 0.9827640458818389] [0.982391626933172, 0.9832925824343233, 0.9825722764958769, 0.9828483951182125, 0.9829111924741615, 0.9827501327627972, 0.9826333606638721, 0.9828159145070102, 0.9824765524733897, 0.9829484249555738]


100%|██████████| 100000/100000 [2:55:32<00:00,  9.49it/s, loss=6.27]    


TypeError: 'NoneType' object is not iterable

In [14]:
adata_ctrl = ad.concat(adata_ctrl_subsetted)
sc.pp.subsample(adata_ctrl, n_obs=300)
adata_ctrl.uns = adata_train.uns.copy()

In [15]:
preds = []
covariate_data = adata_ood_perturbed.obs.drop_duplicates(subset=["condition"])

for i in range(5):
    preds.append(cf.predict(adata=adata_ctrl, sample_rep="X_pca", condition_id_key="condition", rng=jax.random.PRNGKey(i), covariate_data=covariate_data))

    
    
    
    

[########################################] | 100% Completed | 101.63 ms
[########################################] | 100% Completed | 100.60 ms
[########################################] | 100% Completed | 101.64 ms
[########################################] | 100% Completed | 101.19 ms
[########################################] | 100% Completed | 109.15 ms
[########################################] | 100% Completed | 100.87 ms
[########################################] | 100% Completed | 102.96 ms
[########################################] | 100% Completed | 101.16 ms
[########################################] | 100% Completed | 101.58 ms
[########################################] | 100% Completed | 101.09 ms


In [16]:
import pickle
adatas_preds_all = []
for i, pred in enumerate(preds):
    adata_preds = []
    for cond, array in pred.items():
    
        obs_data = pd.DataFrame({
            'condition': [cond] * array.shape[0]
        })
        adata_pred = ad.AnnData(X=np.empty((len(array),adata_train.n_vars)), obs=obs_data)
        adata_pred.obsm["X_pca"] = np.squeeze(array)
        adata_preds.append(adata_pred)
    
    adata_preds = ad.concat(adata_preds)
    adata_preds.obs["sample"] = i
    adata_preds.var_names = adata_train.var_names
    adatas_preds_all.append(adata_preds)

    
adata_pred_all = ad.concat(adatas_preds_all)

with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/degs.pkl", "rb") as pickle_file:
    deg_genes = pickle.load(pickle_file)

with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/idcs_to_keep.pkl", "rb") as pickle_file:
    idcs_to_keep = pickle.load(pickle_file)



    

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/icb/dominik.klein/mambaforge/e

In [18]:
cytokine=cytokine_held_out
adata_full = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/pbmc_with_pca.h5ad")


In [54]:
adata_full.obsm#["X_pca"].shape

AxisArrays with keys: X_pca

In [19]:
with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/idcs_to_keep.pkl", "rb") as pickle_file:
    idcs_to_keep = pickle.load(pickle_file)

adata_ref = adata_full[adata_full.obs_names.isin(idcs_to_keep)]
    

In [20]:
cellflow.preprocessing.reconstruct_pca(query_adata=adata_pred_all, ref_adata=adata_base, use_rep="X_pca", layers_key_added = "X_recon")
adata_pred_all.X = adata_pred_all.layers["X_recon"]
project_pca(query_adata=adata_pred_all, ref_adata=adata_ref, obsm_key_added="X_pca_for_ct_transfer")
project_pca(query_adata=adata_pred_all, ref_adata=adata_full, obsm_key_added="X_pca")
    

In [40]:
#adata_ood_perturbed.obs["cell_type_new"] = adata_ood_perturbed.obs["cell_type"]

In [37]:
project_pca(query_adata=adata_ood_perturbed, ref_adata=adata_full, obsm_key_added="X_pca")


/ictstr01/home/icb/dominik.klein/git_repos/cell_flow_perturbation/src/cellflow/preprocessing/_pca.py:201: ImplicitModificationWarning: Setting element `.obsm['X_pca']` of view, initializing view as actual.
  query_adata.obsm[obsm_key_added] = np.array((X - np.transpose(np.array(ref_means))) @ np.array(ref_pcs))


In [52]:
# TODO: frechet variance in orig 100dim space, not in eval space as we don't have access to it

distr_dicts = {}
gex_dicts = {}
out_dicts = {}
for condition in adata_pred_all.obs["condition"].unique():
    cytokine=cytokine_held_out
    donor=condition.split("_")[0]
    adata_ood_true_cond = adata_ood_perturbed[(adata_ood_perturbed.obs["donor"] == donor) & (adata_ood_perturbed.obs["cytokine"]==cytokine)]
    donor_deg_dict = {k: v for k, v in deg_genes.items() if (k.startswith(donor) and k.endswith(f"_{cytokine}"))}
    adata_pred = adata_pred_all[adata_pred_all.obs["condition"]==condition]
    adata_pred.uns["donors_in_train"] = list(adata_to_append.obs["donor"].unique())
    cond_orig = condition
    #adata_pred.write_h5ad(os.path.join(config_dict["training"]["out_dir"], f"{wandb.run.name}_{condition}_preds.h5ad"))
    out, distr_dict, gex_dict = compute_metrics(adata_ref=adata_ref, adata_pred_all_samples=adata_pred, donor_deg_dict=donor_deg_dict, adata_ood_true=adata_ood_true_cond, adata_ctrl=adata_ctrl)
    distr_dicts[condition] = distr_dict
    out_dicts[condition] = out
    gex_dicts[condition] = gex_dict
    #pd.DataFrame.from_dict(out, columns=[condition], orient="index").to_csv(os.path.join(config_dict["training"]["out_dir"], f"{wandb.run.name}_{condition}.csv"))
    #wandb.log({condition: out})


/tmp/ipykernel_4032555/3245858440.py:12: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata_pred.uns["donors_in_train"] = list(adata_to_append.obs["donor"].unique())
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


0
28 5360
1
28 5360
2
28 5360
3
28 5360
4
28 5360


/tmp/ipykernel_4032555/3245858440.py:12: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata_pred.uns["donors_in_train"] = list(adata_to_append.obs["donor"].unique())
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


0
18 5750
1
18 5750
2
18 5750
3
18 5750
4
18 5750


/tmp/ipykernel_4032555/3245858440.py:12: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata_pred.uns["donors_in_train"] = list(adata_to_append.obs["donor"].unique())
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


...

0
19 7011
1
19 7011
2
19 7011
3
19 7011
4
19 7011


/tmp/ipykernel_4032555/3245858440.py:12: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata_pred.uns["donors_in_train"] = list(adata_to_append.obs["donor"].unique())
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


0
29 7295
1
29 7295
2
29 7295
3
29 7295
4
29 7295


/tmp/ipykernel_4032555/3245858440.py:12: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata_pred.uns["donors_in_train"] = list(adata_to_append.obs["donor"].unique())
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


0
28 4816
1
28 4816
2
28 4816
3
28 4816
4
28 4816


In [56]:
out_dicts

{'Donor10_TWEAK': {'ood_e_distance': 35.5447244488651,
  'decoded_ood_r_squared': 0.919957531366401,
  'mean_decoded_r_sq_per_cell_type': 0.4521643511349637,
  'mean_e_distance_per_cell_type': 195.72721980231208,
  'mean_deg_r_sq_per_cell_type': -0.04257476681577403,
  'frechet_variance': 0.001247302391758072,
  'var_mean_gex': 1.5596884899287575e-05,
  'frechet_variance_CD4 Naive': 0.02306848429979835,
  'frechet_variance_CD56-dim NK': 0.050769653365612,
  'frechet_variance_CD14 Mono': 6.467025118162444,
  'frechet_variance_CD16 Mono': 18.259132944722147,
  'frechet_variance_CD4 Memory': 2245.8839704904494,
  'frechet_variance_B Intermediate/Memory': 0.07746515787512089,
  'frechet_variance_CD8 Naive': 7432.297164195087,
  'frechet_variance_NKT': 0.09304517368101095,
  'frechet_variance_CD56-bright NK': 0.08539655023361499,
  'frechet_variance_B Naive': 0.031565270771673454,
  'frechet_variance_CD8 Memory': 0.01565083063008546,
  'mean_frechet_variance_per_cell_type': 882.116750351752

In [58]:
df = pd.DataFrame(columns=["e_distance", "e_distance_var", "r_sq_per_cell_type", "r_sq_per_cell_type_var", "deg_r_sq", "deg_r_sq_var"])
   

In [61]:
for i, (k, v) in enumerate(out_dicts.items()):
    df.loc[i, :] = {
        "e_distance": v["ood_e_distance"],
        "e_distance_var": v["frechet_variance"],
        "r_sq_per_cell_type": v["mean_decoded_r_sq_per_cell_type"],
        "r_sq_per_cell_type_var": v["mean_var_mean_gex_per_cell_type"],
        "deg_r_sq": v["mean_deg_r_sq_per_cell_type"],
        "deg_r_sq_var": v["mean_var_mean_deg_gex_per_cell_type"],
    }


In [68]:
df

,e_distance,e_distance_var,r_sq_per_cell_type,r_sq_per_cell_type_var,deg_r_sq,deg_r_sq_var,neg_deg_r_sq,neg_r_sq_per_cell_type
0,35.544724,0.001247,0.452164,0.003108,-0.042575,0.032006,1.042575,0.996892
1,46.854766,0.002283,0.214496,0.002783,-0.565733,0.032577,1.565733,0.997217
2,22.260514,0.00377,0.380341,0.002738,-0.010026,0.010363,1.010026,0.997262
3,26.909146,0.002786,0.683281,0.000092,0.335534,0.000484,0.664466,0.999908
4,12.281725,0.002165,0.601992,0.00044,0.214346,0.001822,0.785654,0.99956
5,55.045624,0.001111,0.60752,0.002018,0.227036,0.020593,0.772964,0.997982
6,47.18849,0.003167,0.256626,0.003763,-0.171217,0.033352,1.171217,0.996237
7,84.496188,0.001133,0.551833,0.000087,0.169435,0.00044,0.830565,0.999913
8,56.719115,0.002674,0.552781,0.000992,0.347348,0.007077,0.652652,0.999008


In [63]:
df["neg_deg_r_sq"] = 1-df["deg_r_sq"]
df["neg_r_sq_per_cell_type"] = 1-df["r_sq_per_cell_type_var"]

In [72]:
cal_log_dict = {}
cal_log_dict["e_dist_calibration"] = df[["e_distance", "e_distance_var"]].corr(method="spearman").iloc[0,1]
cal_log_dict["gex_calibration"] = df[["r_sq_per_cell_type", "r_sq_per_cell_type_var"]].corr(method="spearman").iloc[0,1]
cal_log_dict["deg_calibration"] = df[["neg_deg_r_sq", "deg_r_sq_var"]].corr(method="spearman").iloc[0,1]

In [73]:
cal_log_dict

{'e_dist_calibration': -0.4,
 'gex_calibration': -0.7,
 'deg_calibration': 0.6833333333333333}

In [50]:
df_tmp = pd.DataFrame.from_dict(out, columns=[condition], orient="index")
df_tmp["Donor"]

,Donor9_TWEAK
ood_e_distance,56.719115
decoded_ood_r_squared,0.831690
mean_decoded_r_sq_per_cell_type,0.552781
mean_e_distance_per_cell_type,220.049060
mean_deg_r_sq_per_cell_type,0.347348
frechet_variance,NaN
var_mean_gex,NaN
frechet_variance_CD8 Memory,133.950226
frechet_variance_CD8 Naive,0.011099
frechet_variance_CD4 Memory,0.007816


In [45]:
adata_ood_true_cond.X = adata_ood_true_cond.X.toarray()

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/anndata.py:617: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
  warnings.warn(msg, FutureWarning, stacklevel=1)


In [23]:
adata_pred.obs["condition"].value_counts()

condition
Donor10_TWEAK    140
Name: count, dtype: int64

In [24]:
adata_pred.obs["sample"].value_counts()

sample
0    28
1    28
2    28
3    28
4    28
Name: count, dtype: int64

In [27]:
adata_ood_true.obs["condition"].value_counts()

condition
Donor9_TWEAK    4816
Name: count, dtype: int64

In [56]:
np.isnan(adata_ood_true.X.toarray()).sum()

0

In [44]:
adata_ood_true

View of AnnData object with n_obs × n_vars = 0 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'condition', 'is_control', 'cell_type_new', 'donor_cell_type'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'donor_one_hot', 'esm2_embeddings', 'hvg', 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
    layers: 'counts'

In [42]:
adata_pred.obs["sample"].value_counts()

sample
0    10000
1    10000
2    10000
3    10000
4    10000
Name: count, dtype: int64

In [21]:
adata_pred_all.obs["sample"].value_counts()

sample
0    90000
1    90000
2    90000
3    90000
4    90000
Name: count, dtype: int64

In [153]:
adata_base.uns

{'donor_one_hot': {'Donor1': array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
  'Donor10': array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
  'Donor11': array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
  'Donor12': array([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.]),
  'Donor2': array([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.]),
  'Donor3': array([0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]),
  'Donor4': array([0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.]),
  'Donor5': array([0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]),
  'Donor6': array([0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.]),
  'Donor7': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]),
  'Donor8': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]),
  'Donor9': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])},
 'esm2_embeddings': {'4-1BBL': array([ 0.06635367, -0.08944552,  0.0325702 , ...,  0.00990983,
         -0.0871339 ,  0.01538398]),
  'ADSF': array([ 0

In [154]:
adata_rest.uns["split_info"].keys() #[idx_given_cytokine]["donors_to_impute"]
    

dict_keys(['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '5', '6', '7', '8', '9'])

In [155]:
adata_rest.uns["split_info"]["0"]

{'donors_to_impute': array(['Donor6', 'Donor9', 'Donor8', 'Donor11', 'Donor10', 'Donor5',
        'Donor7', 'Donor1', 'Donor3', 'Donor2', 'Donor4', 'Donor12'],
       dtype=object),
 'donors_to_train_data': array([], dtype=float64)}

In [161]:
for k in adata_rest.uns["split_info"].keys():
    print(set(adata_rest.uns["split_info"][k]["donors_to_train_data"]) == set(adata_rest2.uns["split_info"][k]["donors_to_train_data"]))

True
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
True
False
True
True
True
True
True
True
True
True
True
False
True
True
True
False
False
False
False
False


In [165]:
k="4"
print(set(adata_rest.uns["split_info"][k]["donors_to_train_data"]))
print(set(adata_rest2.uns["split_info"][k]["donors_to_train_data"]))

{'Donor9', 'Donor10'}
{'Donor7', 'Donor4'}


In [157]:
for k,v in adata_rest.uns["split_info"].items():
    print(len(v["donors_to_train_data"]), len(v["donors_to_impute"]))
#adata_rest.uns["split_info"]["1"]

0 12
1 11
4 8
4 8
4 8
5 7
5 7
5 7
6 6
6 6
6 6
7 5
1 11
7 5
7 5
8 4
8 4
8 4
9 3
9 3
9 3
10 2
10 2
1 11
10 2
11 1
11 1
11 1
11 1
11 1
11 1
11 1
11 1
11 1
2 10
11 1
11 1
11 1
2 10
2 10
3 9
3 9
3 9


In [158]:
cytokine_held_out2 = "VEGF"
idx_given_cytokine = "7"
control_key = "is_control"

adata_rest2 = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_cytokine/adata_rest_{cytokine_held_out2}.h5ad")


In [168]:
for cyto in ['IFN-beta','IFN-epsilon','C5a', '4-1BBL','ADSF','APRIL','BAFF', 'Noggin','OSM','OX40L', 'IL-13','IL-15']:
    adata_rest = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_cytokine/adata_rest_{cyto}.h5ad")
    for k,v in adata_rest.uns["split_info"].items():
        if len(v["donors_to_impute"]) in [11, 12]:
            print(cyto, k)

IFN-beta 0
IFN-beta 1
IFN-beta 2
IFN-beta 3
IFN-epsilon 0
IFN-epsilon 1
IFN-epsilon 2
IFN-epsilon 3
C5a 0
C5a 1
C5a 2
C5a 3
4-1BBL 0
4-1BBL 1
4-1BBL 2
4-1BBL 3
ADSF 0
ADSF 1
ADSF 2
ADSF 3
APRIL 0
APRIL 1
APRIL 2
APRIL 3
BAFF 0
BAFF 1
BAFF 2
BAFF 3
Noggin 0
Noggin 1
Noggin 2
Noggin 3
OSM 0
OSM 1
OSM 2
OSM 3
OX40L 0
OX40L 1
OX40L 2
OX40L 3
IL-13 0
IL-13 1
IL-13 2
IL-13 3
IL-15 0
IL-15 1
IL-15 2
IL-15 3


In [166]:
for k,v in adata_rest2.uns["split_info"].items():
    print(k, len(v["donors_to_impute"]))

0 12
1 11
10 8
11 8
12 8
13 7
14 7
15 7
16 6
17 6
18 6
19 5
2 11
20 5
21 5
22 4
23 4
24 4
25 3
26 3
27 3
28 2
29 2
3 11
30 2
31 1
32 1
33 1
34 1
35 1
36 1
37 1
38 1
39 1
4 10
40 1
41 1
42 1
5 10
6 10
7 9
8 9
9 9


In [167]:
for k,v in adata_rest.uns["split_info"].items():
    print(k, len(v["donors_to_impute"]))

0 12
1 11
10 8
11 8
12 8
13 7
14 7
15 7
16 6
17 6
18 6
19 5
2 11
20 5
21 5
22 4
23 4
24 4
25 3
26 3
27 3
28 2
29 2
3 11
30 2
31 1
32 1
33 1
34 1
35 1
36 1
37 1
38 1
39 1
4 10
40 1
41 1
42 1
5 10
6 10
7 9
8 9
9 9
